# Backtest Analysis

Run backtests, analyze results, compute performance metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

## Load Data and Generate Signals

In [ ]:
from crypto_quant.data.loaders import load_canonical_dataset
from crypto_quant.signals.momentum import MomentumSignal
from crypto_quant.signals.mean_reversion import MeanReversionSignal
from crypto_quant.signals.composite import CompositeSignal

data = load_canonical_dataset()

# Generate composite signal
momentum = MomentumSignal(lookback=5, threshold=0.001)
mean_reversion = MeanReversionSignal(window=20, threshold=1.5)
composite = CompositeSignal(
    signals=[momentum, mean_reversion],
    weights={'Momentum_5': 0.5, 'MeanReversion_20': 0.5}
)
signal = composite.generate(data)

print(f"Data shape: {data.shape}")
print(f"Signal distribution: {signal.value_counts()}")

## Run Backtest

In [ ]:
from crypto_quant.backtest.engine import BacktestEngine

backtest = BacktestEngine(
    data=data,
    signal=signal,
    initial_capital=100000.0,
    maker_fee=0.0002,
    taker_fee=0.0005,
    slippage=0.0002,
)

results = backtest.run()
print("Backtest completed")

## Calculate Metrics

In [ ]:
from crypto_quant.backtest.metrics import MetricsCalculator

# Assuming results['equity_curve'] is available
if 'equity_curve' in results:
    equity_curve = results['equity_curve']
    
    metrics_calc = MetricsCalculator(equity_curve)
    metrics = metrics_calc.calculate_metrics()
    
    print("=== Performance Metrics ===")
    for metric_name, metric_value in metrics.items():
        if 'ratio' in metric_name:
            print(f"{metric_name}: {metric_value:.4f}")
        elif 'volatility' in metric_name or 'return' in metric_name:
            print(f"{metric_name}: {metric_value:.4%}")
        else:
            print(f"{metric_name}: {metric_value:.4f}")

## Plot Equity Curve

In [ ]:
if 'equity_curve' in results:
    equity_curve = results['equity_curve']
    
    plt.figure(figsize=(14, 6))
    plt.plot(equity_curve.index, equity_curve.values, linewidth=1)
    plt.title('Strategy Equity Curve')
    plt.xlabel('Date')
    plt.ylabel('Portfolio Value (USD)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

## Drawdown Analysis

In [ ]:
if 'equity_curve' in results:
    equity_curve = results['equity_curve']
    
    # Calculate drawdown
    cummax = equity_curve.cummax()
    drawdown = (equity_curve - cummax) / cummax
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # Drawdown chart
    axes[0].fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color='red')
    axes[0].plot(drawdown.index, drawdown.values, linewidth=0.5, color='red')
    axes[0].set_title('Underwater Plot (Drawdown)')
    axes[0].set_ylabel('Drawdown (%)')
    axes[0].grid(True)
    
    # Cumulative return
    cumulative_return = (equity_curve / equity_curve.iloc[0] - 1) * 100
    axes[1].plot(cumulative_return.index, cumulative_return.values, linewidth=1)
    axes[1].set_title('Cumulative Return')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Return (%)')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

## Monthly Returns Heatmap

In [ ]:
if 'equity_curve' in results:
    equity_curve = results['equity_curve']
    
    # Calculate monthly returns
    monthly_returns = equity_curve.resample('M').last().pct_change()
    
    # Reshape for heatmap
    monthly_returns_df = pd.DataFrame({
        'Year': monthly_returns.index.year,
        'Month': monthly_returns.index.month,
        'Return': monthly_returns.values
    })
    
    pivot_table = monthly_returns_df.pivot_table(
        values='Return',
        index='Year',
        columns='Month'
    )
    
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_table * 100, annot=True, fmt='.1f', cmap='RdYlGn', center=0, 
                cbar_kws={'label': 'Return (%)'})
    plt.title('Monthly Returns Heatmap (%)')
    plt.xlabel('Month')
    plt.ylabel('Year')
    plt.tight_layout()
    plt.show()